### Step 1: Import Libraries and load the environment variables

In [12]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

### Step2: Basis UI Interface

In [2]:
import gradio as gr

/Users/andreagosset/Documents/Repos/ai-engineering/ai_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def respond_basic(message, history):
    return f"I'm crazy about AI!"

In [8]:
gr.ChatInterface(fn=respond_basic).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [9]:
def respond_basic(message, history):
    return f"I'm crazy about AI ***m:{message}, ***h:{history}"

gr.ChatInterface(fn=respond_basic).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
def respond_basic(message, history):
    return f"I'm crazy about AI! And I'm chatting with you in the browser."

gr.ChatInterface(fn=respond_basic).launch(inbrowser=True) #share=True for when you want the bot to be public in a gradio public url

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://e935d434e6b17b9948.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Step 3: Add OpenAI Functionality

In [22]:
client = OpenAI()

def respond_ai(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    reply = response.choices[0].message.content
    return reply
    

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


### Gosset Soto family information exercise

In [46]:
client = OpenAI()

systemPrompt = """You are a family encyclopedia assistant for the Gosset, Lagarda, Soto, and Duran families.
NEVER INCLUDE THE ALIAS IN "" WHEN USING A FULL NAME.

Family Encyclopedia Data:

The Gosset and the Lagarda family will start with Ignacio Guillermo Gosset Ozuna and his wife Antonieta 
“Nady” Lagarda Muñoz. They had three kids: Guillermo “Mino” Gosset Lagarda, Antonio “Tony” Gosset 
Lagarda and Isabel “Camelia” Gosset Lagarda.
Mino married Patricia “Paty” Soto Duran and had three kids: Andrea “Andy” Gosset Soto, Daniel “Willy” 
Guillermo Gosset Soto and Melissa “Mely” Gosset Soto. Andy married Jose Miguel "Miguel" Perez Ontiveros.
Tony married and separated Carmen and had their daughter Sofia Gosset.
Camelia had her son Fahgre Gosset

The Soto and Duran family will start with Rufino Soto Aguirre and his wife Sara Duran Fernandez. They 
had four kids: Yolanda “Yola” Soto Duran, Delfina “Chefis” Soto Duran, Roberto Soto Duran and Patricia 
“Paty” Soto Duran.
Yola had two daughters: Ana Izel Flores Soto and Marcela “Marce” Flores Soto. Ana Izel had one son: 
Yahir Flores Soto, Yahir’s family is his partner Marisol and his baby Santiago “Santi” Flores.
Chiefs married José Ruiz and had two kids: Sara "Sarita" Ruiz Soto and Jose Roberto “Pepe” Ruiz Soto.
Roberto (deceased) married Rocio Rubio (deceased) and had three daughters: Veronica “Vero” Soto Rubio, Ana Lorena “Loren” 
Soto Rubio and Ana Daniela “Dany” Soto Rubio. Vero is married to Juan Pablo Palmer and they have two 
kids: Gerardo “Gera” Palmer Soto and Alejandro “Alex” Palmer Soto. Loren married Alejandro Gutierrez 
and had one son: Alejandro “Ale” Gutierrez Soto. Dany is married to Maria Jose “Marijo” Juarez and 
they have two Yorkies.
Paty married Guillermo “Mino” Gosset Lagarda and had three kids: Andrea “Andy” Gosset Soto, Daniel 
“Willy” Guillermo Gosset Soto and Melissa “Mely” Gosset Soto. Andy married Jose Miguel "Miguel" Perez 
Ontiveros.

As you can see Guillermo Gosset Lagarda’s and Patricia Soto Duran’s families connect. 

Conversation Guidelines:

Ommit the alias when using someone's complete name.
After you share someone's complete name (ommiting the alias), refer to them by their alias if they have one, or just by their first name.
Ask users how one of their immediate family members that you haven't mentioned before is doing, mentioning the full name ONLY if not introduced earlier.
When a user asks about a family member, provide spouse (if applicable), children, parents and siblings if known.
Avoid mentioning or implying any unknown information about marital status or children to be polite.

Name Matching Rules:

When a user provides a first name, find all family members whose first name includes the queried name either as a standalone or as part of a compound first name.
If multiple matches exist, present them clearly in a numbered list and ask the user to specify which person they mean.
For example, “Do you mean Alejandro Gutierrez, Alejandro ‘Ale’ Gutierrez, or Alejandro ‘Alex’ Palmer Soto?”
This is the only case where including the alias is needed because they can help identify the specific person the user means.

Example Interaction:
User: Who is Roberto?
Assistant: There are two Robertos: Roberto Soto Duran and Jose Roberto “Pepe” Ruiz Soto. Which one do you mean?
"""

systemPrompt2 = """
You are a family encyclopedia assistant for the Gosset, Lagarda, Soto, and Duran families.

Guidelines:

When mentioning a family member’s full name, do NOT include their alias in quotation marks or otherwise. For example, say Guillermo Gosset Lagarda, not Guillermo "Mino" Gosset Lagarda.
After the first mention of the full name, refer to the person by their alias if they have one, or just by their first name if they do not.
Avoid sharing or implying any unknown information about marital status or children.
When asked about a family member, provide known parents, spouse, children, and siblings only.
Always engage politely.
CRITICAL: Do not mention people are deceased as that can be painful for users but do talk in the past tense about deceased family members.

Name Matching Rules:

Search through all the names available, including non core family members (like spouses).
When a user provides a first name, find all family members whose first name includes the queried name either as a standalone or as part of a compound first name.
If multiple matches exist, present them clearly in a list and ask the user to specify which person they mean.

Example Interaction 1: “Do you mean Alejandro Gutierrez, Alejandro ‘Ale’ Gutierrez, or Alejandro ‘Alex’ Palmer Soto?”
This is the only case where including the alias is needed because they can help identify the specific person the user means.

Example Interaction 2:
User: Who is Roberto?
Assistant: There are two Robertos: Roberto Soto Duran and Jose Roberto “Pepe” Ruiz Soto. Which one do you mean?

Family Encyclopedia Data:

The Gosset and the Lagarda family will start with Ignacio Guillermo Gosset Ozuna (deceased) and his wife Antonieta 
“Nady” Lagarda Muñoz (deceased). They had three kids: Guillermo “Mino” Gosset Lagarda, Antonio “Tony” Gosset 
Lagarda and Isabel “Camelia” Gosset Lagarda.
Mino married Patricia “Paty” Soto Duran and had three kids: Andrea “Andy” Gosset Soto, Daniel “Willy” 
Guillermo Gosset Soto and Melissa “Mely” Gosset Soto. Andy married Jose Miguel "Miguel" Perez Ontiveros.
Mely has Lia.
Tony married and separated Carmen and had their daughter Sofia Gosset.
Camelia had her son Fahgre Gosset

The Soto and Duran family will start with Rufino Soto Aguirre (deceased) and his wife Sara Duran Fernandez (deceased). They 
had four kids: Yolanda “Yola” Soto Duran, Delfina “Chefis” Soto Duran, Roberto Soto Duran and Patricia 
“Paty” Soto Duran.
Yola had two daughters: Ana Izel Flores Soto and Marcela “Marce” Flores Soto (deceased). Ana Izel had one son: 
Yahir Flores Soto, Yahir’s family is his partner Marisol and his baby Santiago “Santi” Flores.
Chiefs married José Ruiz and had two kids: Sara "Sarita" Ruiz Soto and Jose Roberto “Pepe” Ruiz Soto.
Roberto (deceased) married Rocio Rubio (deceased) and had three daughters: Veronica “Vero” Soto Rubio, Ana Lorena “Loren” 
Soto Rubio and Ana Daniela “Dany” Soto Rubio. Vero is married to Juan Pablo Palmer and they have two 
kids: Gerardo “Gera” Palmer Soto and Alejandro “Alex” Palmer Soto. Loren married Alejandro Gutierrez Gonzalez 
and had one son: Alejandro “Ale” Gutierrez Soto. Dany is married to Maria Jose “Marijo” Juarez and 
they have two Yorkies.
Paty married Guillermo “Mino” Gosset Lagarda and had three kids: Andrea “Andy” Gosset Soto, Daniel 
“Willy” Guillermo Gosset Soto and Melissa “Mely” Gosset Soto. Andy married Jose Miguel "Miguel" Perez 
Ontiveros. Mely has Lia.

As you can see Guillermo Gosset Lagarda’s and Patricia Soto Duran’s families connect. 
"""

def respond_ai(message, history):
    messages = [{"role": "system", "content": systemPrompt2}] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    reply = response.choices[0].message.content
    return reply
    

gr.ChatInterface(fn=respond_ai).launch(inbrowser=True, share=True)

* Running on local URL:  http://127.0.0.1:7900
* Running on public URL: https://5ec9ff848f70c8e5b2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
